# Polynomial regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
When the relation is curved (for example fuel use against speed), a line is not enough. A simple trick: build new features $x^2, x^3, \dots$ and run **the same linear regression** on them:

$$\hat y=b+w_1x+w_2x^2+\dots+w_dx^d$$

The model is curved in $x$ but still linear in the coefficients $w$, so least squares still works.

## How do we choose the degree?
| Degree | Result |
|---|---|
| Too low (1) | **Underfitting**: misses the curve; train and test error both high |
| Suitable (3 to 5 for a sine) | Lowest test error |
| Too high (15) | **Overfitting**: passes through every point; low train error, high test error |

> [Underfitting, a good fit and overfitting.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/overfit.png)

*Underfitting, a good fit and overfitting.*

[[lab:reg_poly]]

In the lab, move the degree from 1 to 15 and write down the train and test RMSE. The train error always goes down, but the test error first falls and then rises. That is the **validation curve**:



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(1)
x = rng.uniform(0, 1, 30); y = np.sin(2 * np.pi * x) + rng.normal(0, 0.25, 30)
xt = rng.uniform(0, 1, 200); yt = np.sin(2 * np.pi * xt) + rng.normal(0, 0.25, 200)

degrees = range(1, 16)
train, test = [], []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x[:, None], y)
    train.append(mean_squared_error(y, m.predict(x[:, None])))
    test.append(mean_squared_error(yt, m.predict(xt[:, None])))

best = degrees[int(np.argmin(test))]
print("Best degree on the test data:", best)
plt.plot(degrees, train, "o-", label="train")
plt.plot(degrees, test, "o-", label="test")
plt.yscale("log"); plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend()
plt.title("Validation curve"); plt.show()



## Several features and products
With two features and degree 2, `PolynomialFeatures` builds $x_1, x_2, x_1^2, x_1x_2, x_2^2$. The term $x_1x_2$ is an **interaction**: the effect of one feature depends on the value of the other.

| Original features | Degree | New columns |
|---|---|---|
| 2 | 2 | 5 |
| 10 | 2 | 65 |
| 10 | 3 | 285 |

> The number of columns grows fast. With a high degree you always need scaling and regularization (Ridge or Lasso, next lessons).



In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[2, 3]])
p = PolynomialFeatures(degree=2, include_bias=False)
print(p.fit_transform(X))
print(p.get_feature_names_out(["x1", "x2"]))
print("10 features, degree 3 ->", PolynomialFeatures(3, include_bias=False).fit(np.zeros((1, 10))).n_output_features_, "columns")



> [Polynomial regression on real data.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/r_poly.png)

*Polynomial regression on real data.*

### Exercise

Use cross_val_score with 5 folds to find the best degree for the same sine data.

In [ ]:
# Your answer here